# AR × pearson — production HStack1024 regression + OOD

This notebook performs clean inference on new compounds from
`cleaned_Casestudy.csv`. It uses the frozen HStack1024 extractor, frozen
MinMaxScaler, frozen SelectKBest selector, and frozen final regressor. It does
not replay historical Test-R2, compare against reference results, or consume
reproduce-00 split-oracle outputs.


## Production workflow: Input → Prediction → OOD

Run target: **AR × pearson**

| Step | Stage | Input | Process | Output |
|---:|---|---|---|---|
| 1 | Runtime | GitHub source + frozen Kaggle assets | Load versioned production runtime | Ready pipeline |
| 2 | Domain reference | Raw AR/ER/GR/PR CSV | Standard cleaning and deterministic 60% train partition; no oracle checks | Training SMILES |
| 3 | Query | `cleaned_Casestudy.csv` (`ID,Smiles`) | Schema and SMILES safety checks | New compounds |
| 4 | Extraction | Train + query SMILES | Fresh CPU SMILES/SELFIES/Graph/Fingerprint inference | HStack1024 |
| 5 | Prediction | Query HStack1024 | Frozen MinMax → frozen SelectKBest → frozen Ridge/ElasticNet | Predicted pIC50 |
| 6 | OOD | Scaled train/query HStack1024 | Professor kNN threshold for every k=3…25 | IND/OOD labels and coverage |
| 7 | Export | Predictions + AD labels | Write CSV/JSON provenance | Production artifacts |

The query file has no measured pIC50. Accordingly, this production run does not
calculate R2, RMSE, MAE, residuals, or historical-reference deltas.


In [ ]:
# Production configuration
RUN_DATASET = 'AR'
RUN_METHOD = 'pearson'

GITHUB_REPO_URL = "https://github.com/bigmanaschai/RegressionPipeline_hstack1024_FS.git"
GITHUB_REF = "main"
PROJECT_ROOT_OVERRIDE = ""
BASE_ROOT_OVERRIDE = ""
FS_ROOT_OVERRIDE = ""
QUERY_CSV = "/kaggle/input/datasets/manaschaiaonon/hstack1024-pipeline-libs/cleaned_Casestudy.csv"
QUERY_ID_COLUMN = "ID"
QUERY_SMILES_COLUMN = "Smiles"
OUTPUT_ROOT_OVERRIDE = ""
OOD_K_VALUES = tuple(range(3, 26))

assert RUN_DATASET in {"AR", "ER", "GR", "PR", "ALL"}
assert RUN_METHOD in {"mutual_info", "pearson", "ALL"}
assert OOD_K_VALUES == tuple(range(3, 26))


In [ ]:
# Resolve or clone the GitHub project without modifying Kaggle Inputs.
from pathlib import Path
import subprocess

EXPECTED_RELATIVE = Path("standard_pipeline/RegressionPipeline_hstack1024_FS")

def find_extension_root(project_root):
    """Support both the monorepo layout and this package's standalone GitHub repo."""
    nested = project_root / EXPECTED_RELATIVE
    if nested.is_dir():
        return nested
    if (
        (project_root / "src" / "hstack1024_fs_pipeline").is_dir()
        and (project_root / "materials" / "hstack1024-fs-extension").is_dir()
        and (project_root / "pyproject.toml").is_file()
    ):
        return project_root
    raise FileNotFoundError(
        f"RegressionPipeline_hstack1024_FS source not found under: {project_root}"
    )

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif (Path.cwd() / EXPECTED_RELATIVE).is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
else:
    if not GITHUB_REPO_URL:
        raise ValueError(
            "Set GITHUB_REPO_URL to the repository pushed from this materials folder, "
            "or set PROJECT_ROOT_OVERRIDE to an existing checkout."
        )
    checkout = Path("/kaggle/working/hstack1024-fs-repository")
    if not checkout.is_dir():
        subprocess.check_call(
            ["git", "clone", "--no-checkout", "--filter=blob:none", GITHUB_REPO_URL, str(checkout)]
        )
    subprocess.check_call(
        ["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", GITHUB_REF]
    )
    subprocess.check_call(
        ["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]
    )
    PROJECT_ROOT = checkout.resolve()

EXTENSION_ROOT = find_extension_root(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Extension   :", EXTENSION_ROOT)


In [ ]:
# Install only missing optional deep-inference dependencies.
import importlib.util
import os
import subprocess
import sys

# These frozen inference pipelines do not use experiment tracking.
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("WANDB_SILENT", "true")

required_modules = {
    "transformers": "transformers==4.48.3",
    "sentencepiece": "sentencepiece",
    "selfies": "selfies",
    "rdkit": "rdkit",
    "deepchem": "deepchem",
    "torch_geometric": "torch-geometric",
    "skfp": "git+https://github.com/plenoi/scikit-finger-plenoi.git@master",
}
missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from transformers import AutoModel, AutoTokenizer

def pin_remote_code_revision(auto_class):
    original = auto_class.from_pretrained
    if getattr(original, "_hstack_revision_pinned", False):
        return
    def pinned(pretrained_model_name_or_path, *args, **kwargs):
        revision = kwargs.get("revision")
        if revision:
            kwargs.setdefault("code_revision", revision)
        return original(pretrained_model_name_or_path, *args, **kwargs)
    pinned._hstack_revision_pinned = True
    auto_class.from_pretrained = pinned

pin_remote_code_revision(AutoTokenizer)
pin_remote_code_revision(AutoModel)
print("Optional dependencies: OK")


In [ ]:
# Resolve packages and verify the small GitHub material bundle byte-for-byte.
import hashlib
import importlib
import json
import sys

BASE_KAGGLE_ROOT = Path("/kaggle/input/datasets/manaschaiaonon/hstack1024-pipeline-libs")
LOCAL_BASE_ROOT = PROJECT_ROOT / "standard_pipeline/frozen_hstack1024_ridge/kaggle_dataset/hstack1024-pipeline-libs"
BASE_ROOT_CANDIDATE = Path(BASE_ROOT_OVERRIDE).expanduser() if BASE_ROOT_OVERRIDE else (
    BASE_KAGGLE_ROOT if BASE_KAGGLE_ROOT.is_dir() else LOCAL_BASE_ROOT
)

def find_base_artifact_root(candidate):
    """Accept direct or one-level/nested Kaggle uploads of the flat bundle."""
    candidate = Path(candidate)
    direct = candidate / "checkpoints" / "model_smiles_AR.pt"
    if direct.is_file():
        return candidate
    roots = sorted({
        path.parent.parent.resolve()
        for path in candidate.rglob("model_smiles_AR.pt")
        if path.parent.name == "checkpoints"
    })
    if len(roots) == 1:
        return roots[0]
    return candidate

BASE_ROOT = find_base_artifact_root(BASE_ROOT_CANDIDATE)
FS_ROOT = Path(FS_ROOT_OVERRIDE).expanduser() if FS_ROOT_OVERRIDE else (
    EXTENSION_ROOT / "materials/hstack1024-fs-extension"
)
BASE_CODE_SRC = FS_ROOT / "base_src"
FS_SRC = FS_ROOT / "src"
if not BASE_ROOT.is_dir():
    raise FileNotFoundError(f"Attach base Kaggle library; root not found: {BASE_ROOT}")
if not (BASE_CODE_SRC / "hstack1024_pipeline").is_dir():
    raise FileNotFoundError(f"Vendored base source package not found: {BASE_CODE_SRC}")
if not (FS_SRC / "hstack1024_fs_pipeline").is_dir():
    raise FileNotFoundError(f"FS source package not found: {FS_SRC}")

manifest = json.loads((FS_ROOT / "MANIFEST.json").read_text(encoding="utf-8"))
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
for relative, metadata in manifest["files"].items():
    path = FS_ROOT / relative
    if not path.is_file() or sha256(path) != metadata["sha256"]:
        raise RuntimeError(f"FS material missing or stale: {relative}")

for module_name in list(sys.modules):
    if (
        module_name == "hstack1024_pipeline"
        or module_name.startswith("hstack1024_pipeline.")
        or module_name == "hstack1024_fs_pipeline"
        or module_name.startswith("hstack1024_fs_pipeline.")
    ):
        del sys.modules[module_name]
sys.path[:0] = [str(FS_SRC), str(BASE_CODE_SRC)]
importlib.invalidate_caches()

import hstack1024_pipeline
import hstack1024_fs_pipeline

expected_imports = {
    "hstack1024_pipeline": (hstack1024_pipeline, BASE_CODE_SRC / "hstack1024_pipeline"),
    "hstack1024_fs_pipeline": (hstack1024_fs_pipeline, FS_SRC / "hstack1024_fs_pipeline"),
}
for package_name, (package, expected_dir) in expected_imports.items():
    imported_file = getattr(package, "__file__", None)
    if imported_file is None or Path(imported_file).resolve().parent != expected_dir.resolve():
        raise RuntimeError(
            f"Imported {package_name} from {imported_file!r}; expected {expected_dir}"
        )

base_config = importlib.import_module("hstack1024_pipeline.config")
print("Base config    :", base_config.__file__)
print(
    "Artifact dirs :",
    {
        directory: (BASE_ROOT / directory).is_dir()
        for directory in ("checkpoints", "models", "transformers")
    },
)
print("Flat bundle    :", base_config.is_flat_kaggle_bundle(BASE_ROOT))

base_contract = getattr(hstack1024_pipeline, "PIPELINE_CONTRACT_VERSION", None)
if base_contract != "fresh-raw-split-extraction-cpu-v2":
    raise RuntimeError(
        "Incompatible base HStack1024 contract: "
        f"{base_contract!r} from {hstack1024_pipeline.__file__}"
    )
fs_contract = getattr(hstack1024_fs_pipeline, "PIPELINE_CONTRACT_VERSION", None)
if fs_contract != "fresh-raw-split-extraction-selectkbest-cpu-v1":
    raise RuntimeError(
        "Incompatible HStack1024 FS contract: "
        f"{fs_contract!r} from {hstack1024_fs_pipeline.__file__}"
    )
print("Base root     :", BASE_ROOT)
print("FS materials  :", FS_ROOT)
print("Inference device: CPU")


# Require the production OOD runtime from the checked-out, manifest-verified source.
ood_source = FS_SRC / "hstack1024_fs_pipeline" / "ood.py"
if not ood_source.is_file():
    raise RuntimeError(
        "Production OOD runtime is absent from this checkout: "
        f"{ood_source}. Restart the session and Run All with the latest GITHUB_REF."
    )
ood_module = importlib.import_module("hstack1024_fs_pipeline.ood")
if Path(ood_module.__file__).resolve() != ood_source.resolve():
    raise RuntimeError(
        f"Imported production OOD runtime from {ood_module.__file__!r}; "
        f"expected {ood_source}"
    )
print("Production OOD runtime:", ood_module.__file__)


In [ ]:
# Resolve only the production inputs and frozen artifacts.
from hstack1024_pipeline.config import DATASETS, FAMILY_ORDER, checkpoint_path, fingerprint_transformer_path, raw_csv_path
from hstack1024_fs_pipeline.config import FS_METHODS, bundle_path, get_variant
from hstack1024_fs_pipeline.ood import load_query_compounds

datasets = tuple(DATASETS) if RUN_DATASET == "ALL" else (RUN_DATASET,)
methods = FS_METHODS if RUN_METHOD == "ALL" else (RUN_METHOD,)
raw_csv_paths = {dataset: raw_csv_path(dataset, BASE_ROOT) for dataset in datasets}
query_csv = Path(QUERY_CSV).expanduser()

if not query_csv.is_file():
    raise FileNotFoundError(f"Production query CSV not found: {query_csv}")
query_preview = load_query_compounds(
    query_csv,
    id_column=QUERY_ID_COLUMN,
    smiles_column=QUERY_SMILES_COLUMN,
)
for dataset in datasets:
    if not raw_csv_paths[dataset].is_file():
        raise FileNotFoundError(raw_csv_paths[dataset])
    for family in FAMILY_ORDER:
        artifact = checkpoint_path(dataset, family, BASE_ROOT)
        if not artifact.is_file():
            raise FileNotFoundError(artifact)
    transformer = fingerprint_transformer_path(dataset, BASE_ROOT)
    if not transformer.is_file():
        raise FileNotFoundError(transformer)
    for method in methods:
        model_bundle = bundle_path(dataset, method, fs_root=FS_ROOT)
        if not model_bundle.is_file():
            raise FileNotFoundError(model_bundle)
        spec = get_variant(dataset, method)
        print("Production variant:", spec.variant_id, spec.model_class, f"FS_k={spec.selected_k}")

print("Query CSV      :", query_csv)
print("Query rows     :", len(query_preview))
print("Query columns  :", list(query_preview.columns))
print("Production inputs: OK")


In [ ]:
# Run frozen prediction and professor-defined OOD analysis on new compounds.
from hstack1024_fs_pipeline.ood import run_production_ood_all, run_production_ood_dataset

target_name = "all" if RUN_DATASET == "ALL" else f"{RUN_DATASET}_{RUN_METHOD}"
default_output = Path("/kaggle/working") / f"hstack1024_production_ood_{target_name}"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser() if OUTPUT_ROOT_OVERRIDE else default_output

if RUN_DATASET == "ALL":
    ood_report = run_production_ood_all(
        query_csv,
        OUTPUT_ROOT,
        datasets=datasets,
        methods=methods,
        base_root=BASE_ROOT,
        fs_root=FS_ROOT,
        repo_root=PROJECT_ROOT,
        raw_csv_paths=raw_csv_paths,
        id_column=QUERY_ID_COLUMN,
        smiles_column=QUERY_SMILES_COLUMN,
    )
else:
    ood_report = run_production_ood_dataset(
        RUN_DATASET,
        query_csv,
        OUTPUT_ROOT,
        methods=methods,
        base_root=BASE_ROOT,
        fs_root=FS_ROOT,
        repo_root=PROJECT_ROOT,
        raw_csv=raw_csv_paths[RUN_DATASET],
        id_column=QUERY_ID_COLUMN,
        smiles_column=QUERY_SMILES_COLUMN,
    )

print("Production output:", OUTPUT_ROOT)
ood_report


In [ ]:
# Plot domain coverage across the mandatory k=3..25 range.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for (dataset, method), frame in ood_report.groupby(["Dataset", "FS_Method"], sort=True):
    ax.plot(
        frame["k"],
        frame["IND_Coverage"],
        marker="o",
        label=f"{dataset}-{method}",
    )
ax.set(title="New-compound applicability-domain coverage", xlabel="k", ylabel="IND coverage")
ax.set_xticks(range(3, 26, 2))
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
figure_path = OUTPUT_ROOT / "ood_coverage_diagnostics.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Coverage figure:", figure_path)


In [ ]:
# Show one production output and list every saved artifact.
prediction_files = sorted(OUTPUT_ROOT.rglob("production_predictions_ood.csv"))
if not prediction_files:
    raise RuntimeError("No production prediction file was written")
sample_predictions = __import__("pandas").read_csv(prediction_files[0])
print("Prediction files:", len(prediction_files))
print("Example file:", prediction_files[0])
display(sample_predictions.head())
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_ROOT))


## Production interpretation

- `Predicted_pIC50` is generated by the frozen final model selected for that
  dataset/method; it is not an experimentally observed value.
- `ADk3`…`ADk25` report `IND` or `OOD` independently at every controlled k.
- `IND_Coverage` summarizes how much of the new dataset lies inside the current
  training domain; OOD status is an applicability warning, not a class label.
- Because `cleaned_Casestudy.csv` has no measured pIC50, no performance metric
  should be inferred from these predictions.
